In [2]:
import re

def extract_images(xml_text):
    matches = re.findall(r'\[\[(?:Arquivo|File):(.*?)\]\]', xml_text)

    clean_names = set()

    for m in matches:
        name = m.split("|")[0].strip()
        clean_names.add(name)

    return clean_names

In [3]:
import requests
import os
import urllib.parse

API = "https://wiki.saude.gov.br/SISREG/api.php"

os.makedirs("images", exist_ok=True)

def get_image_url(filename):
    params = {
        "action": "query",
        "titles": f"File:{filename}",
        "prop": "imageinfo",
        "iiprop": "url",
        "format": "json"
    }

    r = requests.get(API, params=params).json()
    pages = r["query"]["pages"]

    for p in pages.values():
        if "imageinfo" in p:
            return p["imageinfo"][0]["url"]

    return None


def download_images(images):
    for img in images:
        print("Baixando:", img)

        url = get_image_url(img)
        if not url:
            print("❌ Não encontrado:", img)
            continue

        filename = urllib.parse.quote(img)
        data = requests.get(url).content

        with open(f"images/{img}", "wb") as f:
            f.write(data)

        print("✔ salvo:", img)

In [5]:
with open("SISREG-20260415193614.xml", encoding="utf-8") as f:
    xml = f.read()

images = extract_images(xml)

download_images(images)

Baixando: MENU REGULADOR.png
✔ salvo: MENU REGULADOR.png
Baixando: Psr-1.jpg
✔ salvo: Psr-1.jpg
Baixando: Logo Ministerio Saude.png
✔ salvo: Logo Ministerio Saude.png
Baixando: MENU SOLICITANTE 1.png
✔ salvo: MENU SOLICITANTE 1.png
Baixando: Logo-crga.png
✔ salvo: Logo-crga.png
Baixando: ADM MENU.png
✔ salvo: ADM MENU.png
Baixando: Gear-icon.png
✔ salvo: Gear-icon.png
Baixando: GERANDO BPA 4.png
✔ salvo: GERANDO BPA 4.png
Baixando: Task2.png
✔ salvo: Task2.png
Baixando: Task.png
✔ salvo: Task.png
Baixando: Psr console.png
✔ salvo: Psr console.png
Baixando: MENU VIDEOFONISTA.png
✔ salvo: MENU VIDEOFONISTA.png
Baixando: Voltar.png
✔ salvo: Voltar.png
Baixando: Captche SISREG.gif
✔ salvo: Captche SISREG.gif
Baixando: COLAR NO WORD.png
✔ salvo: COLAR NO WORD.png
Baixando: WIN+R.jpg
✔ salvo: WIN+R.jpg
Baixando: PRINT - TECLADO.jpg
✔ salvo: PRINT - TECLADO.jpg
Baixando: Library.png
✔ salvo: Library.png
Baixando: Question-mark.png
✔ salvo: Question-mark.png
Baixando: MENU COORDENADOR.png
✔ sa

In [10]:
from xml.etree import ElementTree as ET

def extract_pages(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()

    pages = []

    for page in root.findall(".//page"):
        title = page.findtext("title")

        text = page.findtext(".//text")
        if text is None:
            text = ""

        pages.append((title, text))

    return pages

In [11]:
pypandoc.convert_text(text, "md", format="mediawiki")

NameError: name 'text' is not defined

In [8]:
import os
import re
import pypandoc
from xml.etree import ElementTree as ET

os.makedirs("md", exist_ok=True)

def extract_pages(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()

    ns = {"mw": "http://www.mediawiki.org/xml/export-0.11/"}

    pages = []

    for page in root.findall("mw:page", ns):
        title = page.find("mw:title", ns).text

        revision = page.find("mw:revision", ns)
        text = revision.find("mw:text", ns).text or ""

        pages.append((title, text))

    return pages


def clean_filename(title):
    return re.sub(r'[\\/*?:"<>|]', "_", title)


def convert_wikitext_to_md(text):
    return pypandoc.convert_text(text, "md", format="mediawiki")


pages = extract_pages("SISREG-20260415193614.xml")

for title, content in pages:
    md = convert_wikitext_to_md(content)

    filename = clean_filename(title) + ".md"

    with open(f"md/{filename}", "w", encoding="utf-8") as f:
        f.write(md)

    print("✔", filename)

In [9]:
import os
import re

IMG_DIR = "/images"

def fix_images(md):
    md = re.sub(r"Arquivo:([^)\]]+)", IMG_DIR + r"/\1", md)
    return md


for file in os.listdir("md"):
    if file.endswith(".md"):
        path = os.path.join("md", file)

        with open(path, encoding="utf-8") as f:
            content = f.read()

        content = fix_images(content)

        with open(path, "w", encoding="utf-8") as f:
            f.write(content)

In [12]:
import os
import re
import pypandoc
from xml.etree import ElementTree as ET

XML_FILE = "SISREG-20260415193614.xml"
OUT_DIR = "md"
IMG_DIR = "/images"

os.makedirs(OUT_DIR, exist_ok=True)


# -----------------------------
# 1. Extrai páginas do XML (robusto)
# -----------------------------
def extract_pages(xml_file):
    with open(xml_file, encoding="utf-8") as f:
        xml = f.read()

    pages = re.findall(
        r"<page>.*?<title>(.*?)</title>.*?<text[^>]*>(.*?)</text>.*?</page>",
        xml,
        re.DOTALL
    )

    return pages


# -----------------------------
# 2. Nome seguro de arquivo
# -----------------------------
def clean_filename(title):
    title = re.sub(r'[\\/*?:"<>|]', "_", title)
    title = title.strip().replace(" ", "_")
    return title


# -----------------------------
# 3. Converte Wikitext → Markdown
# -----------------------------
def convert_wikitext(text):
    if not text or not text.strip():
        return ""

    try:
        return pypandoc.convert_text(
            text,
            "gfm",  # melhor compatibilidade que "md"
            format="mediawiki"
        )
    except Exception as e:
        print("⚠️ erro pandoc:", e)
        return text


# -----------------------------
# 4. Corrige imagens
# -----------------------------
def fix_images(md):
    # [[Arquivo:img.png]] → /images/img.png
    md = re.sub(r"Arquivo:([^)\]\s]+)", IMG_DIR + r"/\1", md)
    return md


# -----------------------------
# 5. Pipeline principal
# -----------------------------
def main():
    pages = extract_pages(XML_FILE)

    print(f"📄 Páginas encontradas: {len(pages)}")

    if len(pages) == 0:
        print("❌ Nenhuma página encontrada. XML pode estar em outro formato.")
        return

    for title, content in pages:
        md = convert_wikitext(content)
        md = fix_images(md)

        filename = clean_filename(title) + ".md"

        # frontmatter do Just the Docs
        md_final = f"""---
title: {title}
nav_order: 1
---

{md}
"""

        with open(os.path.join(OUT_DIR, filename), "w", encoding="utf-8") as f:
            f.write(md_final)

        print("✔", filename)


if __name__ == "__main__":
    main()

📄 Páginas encontradas: 36
⚠️ erro pandoc: No pandoc was found: either install pandoc and add it
to your PATH or or call pypandoc.download_pandoc(...) or
install pypandoc wheels with included pandoc.
✔ ALVO_DO_CAPACITAÇÃO_DO_SISREG.md
⚠️ erro pandoc: No pandoc was found: either install pandoc and add it
to your PATH or or call pypandoc.download_pandoc(...) or
install pypandoc wheels with included pandoc.
✔ ATUALIZAÇÃO_DO_SISREG.md
⚠️ erro pandoc: No pandoc was found: either install pandoc and add it
to your PATH or or call pypandoc.download_pandoc(...) or
install pypandoc wheels with included pandoc.
✔ Administrador_Estadual.md
⚠️ erro pandoc: No pandoc was found: either install pandoc and add it
to your PATH or or call pypandoc.download_pandoc(...) or
install pypandoc wheels with included pandoc.
✔ Administrador_Municipal.md
⚠️ erro pandoc: No pandoc was found: either install pandoc and add it
to your PATH or or call pypandoc.download_pandoc(...) or
install pypandoc wheels with include

In [16]:
import os
import re
import pypandoc

XML_FILE = "SISREG-20260415193614.xml"
OUT_DIR = "docs"
IMG_DIR = "/images"

os.makedirs(OUT_DIR, exist_ok=True)


# -----------------------------
# 1. Extrair páginas do XML
# -----------------------------
def extract_pages(xml_file):
    with open(xml_file, encoding="utf-8") as f:
        xml = f.read()

    pages = re.findall(
        r"<page>.*?<title>(.*?)</title>.*?<text[^>]*>(.*?)</text>.*?</page>",
        xml,
        re.DOTALL
    )

    return pages


# -----------------------------
# 2. Limpa nome de arquivo
# -----------------------------
def clean_filename(title):
    title = re.sub(r'[\\/*?:"<>|]', "_", title)
    return title.strip().replace(" ", "_")


# -----------------------------
# 3. Define pastas (REGRAS)
# -----------------------------
def get_folder(title):
    t = title.upper()

    if "ADMIN" in t:
        return "01_ADMINISTRACAO"
    if "CAPACIT" in t:
        return "02_CAPACITACAO"
    if "ERRO" in t:
        return "03_ERROS"
    if "SISREG" in t:
        return "04_SISREG"
    if "SOLICITA" in t:
        return "05_SOLICITACAO"
    if "ACESSO" in t:
        return "05_SOLICITACAO"
    if "CADAST" in t:
        return "06_CADASTRO"
    if "GLOSSARIO" in t:
        return "07_GLOSSARIO"

    return "99_OUTROS"


# -----------------------------
# 4. Converte Wikitext → Markdown
# -----------------------------
def convert_wikitext(text):
    if not text:
        return ""

    try:
        return pypandoc.convert_text(
            text,
            "gfm",
            format="mediawiki"
        )
    except Exception as e:
        print("⚠️ erro pandoc:", e)
        return text


# -----------------------------
# 5. Corrige imagens
# -----------------------------
def fix_images(md):
    return re.sub(r"Arquivo:([^)\]\s]+)", IMG_DIR + r"/\1", md)


# -----------------------------
# 6. Pipeline principal
# -----------------------------
def main():
    pages = extract_pages(XML_FILE)

    print(f"📄 Páginas encontradas: {len(pages)}")

    if not pages:
        print("❌ Nenhuma página encontrada no XML.")
        return

    for title, content in pages:

        # converte
        md = convert_wikitext(content)
        md = fix_images(md)

        # pasta
        folder = get_folder(title)
        path_dir = os.path.join(OUT_DIR, folder)
        os.makedirs(path_dir, exist_ok=True)

        # arquivo
        filename = clean_filename(title) + ".md"

        # frontmatter (Just the Docs)
        md_final = f"""---
title: {title}
nav_order: 1
---

{md}
"""

        # salva
        with open(os.path.join(path_dir, filename), "w", encoding="utf-8") as f:
            f.write(md_final)

        print("✔", folder, filename)


if __name__ == "__main__":
    main()

📄 Páginas encontradas: 36
✔ 02_CAPACITACAO ALVO_DO_CAPACITAÇÃO_DO_SISREG.md
✔ 04_SISREG ATUALIZAÇÃO_DO_SISREG.md
✔ 01_ADMINISTRACAO Administrador_Estadual.md
✔ 01_ADMINISTRACAO Administrador_Municipal.md
✔ 99_OUTROS Auditor.md
✔ 04_SISREG CENTRAIS_SISREG.md
✔ 99_OUTROS CGRA.md
✔ 03_ERROS CRIAR_ROTEIRO_ERRO_PASSO-A-PASSO.md
✔ 99_OUTROS Coordenador_de_Unidade.md
✔ 04_SISREG DADOS_PARA_A_CONFIGURAÇÃO_E_IMPLANTAÇÃO_DO_SISREG.md
✔ 99_OUTROS DRAC.md
✔ 03_ERROS Erros_de_Exportação_BPA.md
✔ 05_SOLICITACAO Estatísticas_de_Acesso.md
✔ 99_OUTROS Executante.md
✔ 99_OUTROS Executante_Int.md
⚠️ erro pandoc: Pandoc died with exitcode "64" during conversion: Error at (line 4, column 26):
unexpected 's'
{| class=&quot;wikitable sortable&quot;
                         ^

✔ 99_OUTROS GUIA_DE_PORTARIA,_DECRETO_E_LEI_PARA_CENTRAIS.md
✔ 99_OUTROS Glossário.md
✔ 04_SISREG IMPORTAÇÃO_DE_NOVOS_PROCEDIMENTOS_NO_SISREG.md
✔ 04_SISREG MENSAGEM_DE_CAPTCHA_(VOCÊ_É_UM_ROBÔ)_NO_SISREG.md
✔ 99_OUTROS Ministerio_da_Saú

In [17]:
import re

def fix_images(md):
    # captura [[Arquivo:img.png|...]]
    md = re.sub(
        r"\[\[(?:Arquivo|File):(.*?)(\|.*?)?\]\]",
        r"![](/images/\1)",
        md
    )

    # caso Pandoc já tenha deixado só "Arquivo:img.png"
    md = re.sub(
        r"Arquivo:(\S+)",
        r"![](/images/\1)",
        md
    )

    return md

In [19]:
import os
import re
import pypandoc

XML_FILE = "SISREG-20260415193614.xml"
OUT_DIR = "docs"
IMG_DIR = "/images"

os.makedirs(OUT_DIR, exist_ok=True)


# -----------------------------
# 1. Extrai páginas do XML
# -----------------------------
def extract_pages(xml_file):
    with open(xml_file, encoding="utf-8") as f:
        xml = f.read()

    pages = re.findall(
        r"<page>.*?<title>(.*?)</title>.*?<text[^>]*>(.*?)</text>.*?</page>",
        xml,
        re.DOTALL
    )

    return pages


# -----------------------------
# 2. Nome seguro
# -----------------------------
def clean_filename(title):
    title = re.sub(r'[\\/*?:"<>|]', "_", title)
    return title.strip().replace(" ", "_")


# -----------------------------
# 3. Converte Wikitext → Markdown
# -----------------------------
def convert_wikitext(text):
    if not text:
        return ""

    try:
        return pypandoc.convert_text(
            text,
            "gfm",
            format="mediawiki"
        )
    except Exception as e:
        print("⚠️ erro pandoc:", e)
        return text


# -----------------------------
# 4. CORREÇÃO DEFINITIVA DE IMAGENS
# -----------------------------
def fix_images(md):
    # Caso clássico do MediaWiki:
    # [[Arquivo:img.png|30px]] ou [[File:img.png]]
    md = re.sub(
        r"\[\[(?:Arquivo|File):([^|\]]+)(?:\|.*?)?\]\]",
        r"![](/images/\1)",
        md
    )

    # fallback caso sobrar "Arquivo:"
    md = re.sub(
        r"Arquivo:([^\s\)\]]+)",
        r"![](/images/\1)",
        md
    )

    return md


# -----------------------------
# 5. Classificação em pastas
# -----------------------------
def get_folder(title):
    t = title.upper()

    if "ADMIN" in t:
        return "01_ADMINISTRACAO"
    if "CAPACIT" in t:
        return "02_CAPACITACAO"
    if "ERRO" in t:
        return "03_ERROS"
    if "SISREG" in t:
        return "04_SISREG"
    if "SOLIC" in t:
        return "05_SOLICITACAO"
    if "ACESSO" in t:
        return "05_SOLICITACAO"
    if "CADAST" in t:
        return "06_CADASTRO"
    if "GLOSSARIO" in t:
        return "07_GLOSSARIO"

    return "99_OUTROS"


# -----------------------------
# 6. Pipeline principal
# -----------------------------
def main():
    pages = extract_pages(XML_FILE)

    print(f"📄 Páginas encontradas: {len(pages)}")

    if not pages:
        print("❌ XML vazio ou formato inválido.")
        return

    for title, content in pages:

        # converte wikitext → markdown
        md = convert_wikitext(content)

        # corrige imagens (AQUI ESTAVA O PROBLEMA)
        md = fix_images(md)

        # pasta
        folder = get_folder(title)
        path_dir = os.path.join(OUT_DIR, folder)
        os.makedirs(path_dir, exist_ok=True)

        # arquivo
        filename = clean_filename(title) + ".md"

        # frontmatter do Just the Docs
        md_final = f"""---
title: {title}
nav_order: 1
---

{md}
"""

        # salva
        with open(os.path.join(path_dir, filename), "w", encoding="utf-8") as f:
            f.write(md_final)

        print("✔", folder, filename)


if __name__ == "__main__":
    main()

📄 Páginas encontradas: 36
✔ 02_CAPACITACAO ALVO_DO_CAPACITAÇÃO_DO_SISREG.md
✔ 04_SISREG ATUALIZAÇÃO_DO_SISREG.md
✔ 01_ADMINISTRACAO Administrador_Estadual.md
✔ 01_ADMINISTRACAO Administrador_Municipal.md
✔ 99_OUTROS Auditor.md
✔ 04_SISREG CENTRAIS_SISREG.md
✔ 99_OUTROS CGRA.md
✔ 03_ERROS CRIAR_ROTEIRO_ERRO_PASSO-A-PASSO.md
✔ 99_OUTROS Coordenador_de_Unidade.md
✔ 04_SISREG DADOS_PARA_A_CONFIGURAÇÃO_E_IMPLANTAÇÃO_DO_SISREG.md
✔ 99_OUTROS DRAC.md
✔ 03_ERROS Erros_de_Exportação_BPA.md
✔ 05_SOLICITACAO Estatísticas_de_Acesso.md
✔ 99_OUTROS Executante.md
✔ 99_OUTROS Executante_Int.md
⚠️ erro pandoc: Pandoc died with exitcode "64" during conversion: Error at (line 4, column 26):
unexpected 's'
{| class=&quot;wikitable sortable&quot;
                         ^

✔ 99_OUTROS GUIA_DE_PORTARIA,_DECRETO_E_LEI_PARA_CENTRAIS.md
✔ 99_OUTROS Glossário.md
✔ 04_SISREG IMPORTAÇÃO_DE_NOVOS_PROCEDIMENTOS_NO_SISREG.md
✔ 04_SISREG MENSAGEM_DE_CAPTCHA_(VOCÊ_É_UM_ROBÔ)_NO_SISREG.md
✔ 99_OUTROS Ministerio_da_Saú